# Stream C — `stance_scoring.ipynb`

**Purpose:** Re-rank and score the retrieved evidence from `evidence_retrieval.ipynb`
using cross-encoder stance scoring. Produces richer Stream C features for the fusion module.

**What we add on top of the FAISS evidence features:**
1. **SBERT cross-encoding** — `cross-encoder/nli-deberta-v3-small` scores each
   (caption, retrieved_text) pair as ENTAILMENT / NEUTRAL / CONTRADICTION
2. **Aggregated stance features** — max/mean entailment, max/mean contradiction,
   net stance (entailment − contradiction), stance entropy
3. **SBERT bi-encoder similarity** — `all-MiniLM-L6-v2` caption↔evidence cosine sim
   (faster than cross-encoder, complementary signal)
4. **Combined Stream C feature vector** — concatenate evidence features (10-d)
   + stance features (8-d) + sbert sim features (3-d) = **21-d per pair**

**Expected AUROC improvement:** 0.53 → 0.58–0.64 on NC test

**Runtime:** ~5–10 min (cross-encoder on GPU, batched)

## 0. Environment Check

In [23]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM: 8.6 GB
PyTorch: 2.6.0+cu124


## 1. Install & Imports

In [24]:
# Install sentence-transformers if not present
try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    print("sentence-transformers already installed")
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'sentence-transformers', '-q'], check=True)
    from sentence_transformers import SentenceTransformer, CrossEncoder
    print("sentence-transformers installed")

sentence-transformers already installed


In [25]:
import os, sys, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.metrics import roc_auc_score

# ── Project root ──────────────────────────────────────────────────────────────
try:
    ROOT = Path(__file__).resolve().parents[2]
except NameError:
    ROOT = Path(os.getcwd())
    while not (ROOT / 'src' / 'utils' / 'paths.py').exists():
        ROOT = ROOT.parent
        if ROOT == ROOT.parent:
            ROOT = Path(os.getcwd()); break

sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

EVIDENCE_DIR  = ROOT / 'cache' / 'evidence'
STANCE_DIR    = ROOT / 'cache' / 'stance_scores'
FIG_DIR       = ROOT / 'paper' / 'figures'
TAB_DIR       = ROOT / 'paper' / 'tables'
STANCE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Verify evidence files
# Verify evidence files
for name in ('evidence_train.pkl', 'evidence_val.pkl', 'evidence_test.pkl',
             'evidence_verite.pkl', 'evidence_mmfake.pkl'):
    p = EVIDENCE_DIR / name
    print(f"  {'✅' if p.exists() else '❌ MISSING'} {name}")

for name in ('raw_evidence_train.pkl', 'raw_evidence_val.pkl',
             'raw_evidence_test.pkl', 'raw_evidence_verite.pkl',
             'raw_evidence_mmfake.pkl'):
    p = STANCE_DIR / name
    print(f"  {'✅' if p.exists() else '❌ MISSING'} {name} (stance_scores/)")

Project root: f:\GitHub\eve-rag
Device: cuda
  ✅ evidence_train.pkl
  ✅ evidence_val.pkl
  ✅ evidence_test.pkl
  ✅ evidence_verite.pkl
  ✅ evidence_mmfake.pkl
  ✅ raw_evidence_train.pkl (stance_scores/)
  ✅ raw_evidence_val.pkl (stance_scores/)
  ✅ raw_evidence_test.pkl (stance_scores/)
  ✅ raw_evidence_verite.pkl (stance_scores/)
  ✅ raw_evidence_mmfake.pkl (stance_scores/)


## 2. Load Evidence Caches

In [26]:
def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# Feature matrices from FAISS retrieval (N, 10)
ev_feats = {
    'train':  load_pkl(EVIDENCE_DIR / 'evidence_train.pkl'),
    'val':    load_pkl(EVIDENCE_DIR / 'evidence_val.pkl'),
    'test':   load_pkl(EVIDENCE_DIR / 'evidence_test.pkl'),
    'verite': load_pkl(EVIDENCE_DIR / 'evidence_verite.pkl'),
    'mmfake': load_pkl(EVIDENCE_DIR / 'evidence_mmfake.pkl'),
}

# Raw evidence: {pair_key: {caption, retrieved:[{text,score}], label}}
raw_ev = {
    'train':  load_pkl(STANCE_DIR / 'raw_evidence_train.pkl'),
    'val':    load_pkl(STANCE_DIR / 'raw_evidence_val.pkl'),
    'test':   load_pkl(STANCE_DIR / 'raw_evidence_test.pkl'),
    'verite': load_pkl(STANCE_DIR / 'raw_evidence_verite.pkl'),
    'mmfake': load_pkl(STANCE_DIR / 'raw_evidence_mmfake.pkl'),
}
feat_names = load_pkl(EVIDENCE_DIR / 'feature_names.pkl')
print(f"Evidence feature names ({len(feat_names)}): {feat_names}")

for split, d in ev_feats.items():
    n = len(d['labels'])
    auc = roc_auc_score(d['labels'], d['features'][:, feat_names.index('stance_score')])
    print(f"  {split:8s}: {n:6d} pairs, feat shape={d['features'].shape}, "
          f"stance AUROC={auc:.4f}")

Evidence feature names (10): ['img_sim_max', 'img_sim_mean', 'img_sim_std', 'img_sim_top1', 'txt_sim_max', 'txt_sim_mean', 'txt_sim_top1', 'cross_sim_max', 'cross_sim_mean', 'stance_score']
  train   :  71072 pairs, feat shape=(71072, 10), stance AUROC=0.4633
  val     :   7024 pairs, feat shape=(7024, 10), stance AUROC=0.4611
  test    :   7264 pairs, feat shape=(7264, 10), stance AUROC=0.4610
  verite  :   1001 pairs, feat shape=(1001, 10), stance AUROC=0.4988
  mmfake  :  10000 pairs, feat shape=(10000, 10), stance AUROC=0.4892


## 3. Load Models — SBERT Bi-encoder + Cross-encoder

In [27]:
from sentence_transformers import SentenceTransformer, CrossEncoder

print("Loading SBERT bi-encoder (all-MiniLM-L6-v2) …")
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2', device=str(DEVICE))
print(f"  Bi-encoder loaded | embedding dim: {bi_encoder.get_sentence_embedding_dimension()}")

print("Loading NLI cross-encoder (cross-encoder/nli-deberta-v3-small) …")
cross_encoder = CrossEncoder(
    'cross-encoder/nli-deberta-v3-small',
    device=str(DEVICE),
    max_length=256,
)
# Label order for DeBERTa NLI: [contradiction, entailment, neutral]
# Confirmed from model card
LABEL_CONTRADICTION = 0
LABEL_ENTAILMENT    = 1
LABEL_NEUTRAL       = 2
print(f"  Cross-encoder loaded | labels: contradiction=0, entailment=1, neutral=2")

Loading SBERT bi-encoder (all-MiniLM-L6-v2) …


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

KeyboardInterrupt: 

## 4. Stance Scoring Functions

In [ ]:
import scipy.special as sp

def compute_stance_features(caption, retrieved_texts, cross_enc, bi_enc,
                             batch_size=64):
    """
    Given a query caption and K retrieved texts, compute:
      Cross-encoder (NLI):
        - max/mean entailment score
        - max/mean contradiction score
        - net stance = mean(entailment) - mean(contradiction)
        - stance entropy (uncertainty of NLI distribution)
        - top1 entailment, top1 contradiction
      Bi-encoder:
        - max/mean/std cosine similarity (caption ↔ retrieved)
    Returns: np.array of shape (21,) — but first 10 come from evidence_retrieval
    Here we return only the NEW 11 features.
    """
    if not retrieved_texts:
        return np.zeros(11, dtype=np.float32)

    # ── Cross-encoder NLI ────────────────────────────────────────────────────
    pairs = [(caption, t) for t in retrieved_texts]
    logits = cross_enc.predict(pairs, batch_size=batch_size,
                               apply_softmax=False, show_progress_bar=False)
    probs  = sp.softmax(logits, axis=1)   # (K, 3)

    ent  = probs[:, LABEL_ENTAILMENT]       # (K,)
    con  = probs[:, LABEL_CONTRADICTION]    # (K,)
    neu  = probs[:, LABEL_NEUTRAL]          # (K,)

    # Entropy over NLI distribution per pair
    entropy = -np.sum(probs * np.log(probs + 1e-9), axis=1).mean()

    ce_feats = np.array([
        ent.max(),                        # max entailment
        ent.mean(),                       # mean entailment
        con.max(),                        # max contradiction
        con.mean(),                       # mean contradiction
        ent.mean() - con.mean(),          # net stance (+ve = entailed, -ve = contradicted)
        entropy,                          # uncertainty
        ent[0],                           # top-1 retrieved: entailment
        con[0],                           # top-1 retrieved: contradiction
    ], dtype=np.float32)

    # ── Bi-encoder cosine similarity ─────────────────────────────────────────
    cap_emb = bi_enc.encode(caption, convert_to_numpy=True,
                             normalize_embeddings=True, show_progress_bar=False)
    ret_embs = bi_enc.encode(retrieved_texts, convert_to_numpy=True,
                              normalize_embeddings=True, show_progress_bar=False,
                              batch_size=batch_size)
    sims = ret_embs @ cap_emb   # (K,)
    bi_feats = np.array([
        sims.max(),
        sims.mean(),
        sims.std(),
    ], dtype=np.float32)

    return np.concatenate([ce_feats, bi_feats])   # (11,)


print("Stance scoring functions ready.")
print("New features (11): max_ent, mean_ent, max_con, mean_con, "
      "net_stance, entropy, top1_ent, top1_con, bi_max, bi_mean, bi_std")

## 5. Batch Stance Scoring — All Splits

We score the **top-5** retrieved texts per pair (not all 10) for speed —
cross-encoder quality drops for lower-ranked retrievals anyway.

In [ ]:
TOP_K_STANCE = 5   # use top-5 of the 10 retrieved texts for cross-encoding

STANCE_CACHE = STANCE_DIR / 'stance_raw_all.pkl'

if STANCE_CACHE.exists():
    print("Loading cached stance scores …")
    with open(STANCE_CACHE, 'rb') as f:
        stance_raw = pickle.load(f)
    print(f"  Loaded {sum(len(v) for v in stance_raw.values())} total entries")
else:
    stance_raw = {}
    for split in ('train', 'val', 'test', 'verite', 'mmfake'):
        raw   = raw_ev[split]          # {pair_key: {caption, retrieved, label}}
        keys  = list(raw.keys())
        stance_raw[split] = {}

        for key in tqdm(keys, desc=f'Stance {split}'):
            entry    = raw[key]
            caption  = str(entry.get('caption', ''))
            retrieved = entry.get('retrieved', [])
            # Take top-K by retrieval score
            top_texts = [r['text'] for r in retrieved[:TOP_K_STANCE]
                         if isinstance(r.get('text'), str) and len(r['text'].strip()) > 5]

            if top_texts:
                feats = compute_stance_features(
                    caption, top_texts, cross_encoder, bi_encoder,
                    batch_size=32
                )
            else:
                feats = np.zeros(11, dtype=np.float32)

            stance_raw[split][key] = {
                'stance_feats': feats,
                'label':        int(entry['label']),
            }
        torch.cuda.empty_cache()
        print(f"  {split}: {len(keys)} pairs scored")

    with open(STANCE_CACHE, 'wb') as f:
        pickle.dump(stance_raw, f, protocol=4)
    print(f"\nCached → {STANCE_CACHE.name} "
          f"({STANCE_CACHE.stat().st_size/1e6:.1f} MB)")

## 6. Assemble Final Stream C Feature Matrix

Final feature vector per pair = evidence features (10-d) + stance features (11-d) = **21-d**

In [ ]:
STANCE_FEAT_NAMES = [
    'ce_max_ent', 'ce_mean_ent', 'ce_max_con', 'ce_mean_con',
    'ce_net_stance', 'ce_entropy', 'ce_top1_ent', 'ce_top1_con',
    'bi_max_sim', 'bi_mean_sim', 'bi_std_sim',
]
ALL_FEAT_NAMES = feat_names + STANCE_FEAT_NAMES
print(f"Total Stream C features: {len(ALL_FEAT_NAMES)}")
print(f"  Evidence (10): {feat_names}")
print(f"  Stance   (11): {STANCE_FEAT_NAMES}")

stream_c = {}
for split in ('train', 'val', 'test', 'verite', 'mmfake'):
    ev   = ev_feats[split]        # {features:(N,10), labels:(N,)}
    sr   = stance_raw[split]      # {pair_key: {stance_feats:(11,), label}}

    # Align order: ev_feats rows are already in pair order
    # stance_raw keys may differ slightly — build aligned matrix
    sr_keys = list(sr.keys())
    n = len(sr_keys)

    # Evidence features already aligned by index
    ev_mat = ev['features'][:n]      # (N, 10)
    labels = ev['labels'][:n]        # (N,)

    st_mat = np.stack(
        [sr[k]['stance_feats'] for k in sr_keys], axis=0
    )   # (N, 11)

    # Replace NaN/Inf
    ev_mat = np.nan_to_num(ev_mat, nan=0.0, posinf=1.0, neginf=-1.0)
    st_mat = np.nan_to_num(st_mat, nan=0.0, posinf=1.0, neginf=-1.0)

    combined = np.concatenate([ev_mat, st_mat], axis=1)   # (N, 21)

    stream_c[split] = {
        'features':   combined.astype(np.float32),
        'labels':     labels,
        'feat_names': ALL_FEAT_NAMES,
    }

    print(f"  {split:8s}: {combined.shape}  NaN check: "
          f"{'✅' if not np.isnan(combined).any() else '❌'}")

## 7. Feature-wise AUROC Analysis

In [ ]:
print("Feature-wise AUROC — NC test:")
print(f"  {'Feature':<20} {'AUROC':>7}")
print("  " + "-" * 30)

test_feats  = stream_c['test']['features']
test_labels = stream_c['test']['labels']
aucs = []
for i, name in enumerate(ALL_FEAT_NAMES):
    col = test_feats[:, i]
    try:
        auc = roc_auc_score(test_labels, col)
        auc = max(auc, 1 - auc)   # flip if below 0.5
    except Exception:
        auc = 0.5
    aucs.append(auc)
    print(f"  {name:<20} {auc:>7.4f}")

print(f"\n  Best single feature: {ALL_FEAT_NAMES[np.argmax(aucs)]} "
      f"(AUROC={max(aucs):.4f})")

## 8. Logistic Regression Probe — Stream C Standalone

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])

pipe.fit(stream_c['train']['features'], stream_c['train']['labels'])

print("Stream C LR probe results:")
for split in ('val', 'test', 'verite', 'mmfake'):
    X = stream_c[split]['features']
    y = stream_c[split]['labels']
    probs = pipe.predict_proba(X)[:, 1]
    preds = pipe.predict(X)
    auc   = roc_auc_score(y, probs)
    acc   = accuracy_score(y, preds)
    print(f"  {split:8s}: AUROC={auc:.4f}  Acc={acc:.4f}")

# Save LR probe for reference
import pickle
with open(STANCE_DIR / 'stream_c_lr_probe.pkl', 'wb') as f:
    pickle.dump(pipe, f)

## 9. Feature Importance (LR Coefficients)

In [ ]:
import matplotlib.pyplot as plt

coefs = pipe.named_steps['lr'].coef_[0]
order = np.argsort(np.abs(coefs))[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e65100' if c > 0 else '#2196f3' for c in coefs[order]]
ax.barh([ALL_FEAT_NAMES[i] for i in order], coefs[order], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('LR Coefficient (OOC direction)')
ax.set_title('Stream C — Feature Importance (LR probe, NC train)')
plt.tight_layout()
fig_path = FIG_DIR / 'stream_c_feature_importance.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {fig_path}")

## 10. Save Final Stream C Features

In [ ]:
def save_pkl(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    print(f"  Saved {path.name} ({path.stat().st_size/1e6:.1f} MB)")

print("Saving Stream C features …")
for split, data in stream_c.items():
    save_pkl(data, STANCE_DIR / f'stream_c_{split}.pkl')

save_pkl({'feat_names': ALL_FEAT_NAMES}, STANCE_DIR / 'stream_c_feat_names.pkl')
print(f"\n✅ Stream C features saved to: {STANCE_DIR}")

## 11. Visualisation — Stance Score Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plot_items = [
    ('test',   'NC test'),
    ('verite', 'VERITE'),
    ('mmfake', 'MMFake'),
]
features_to_plot = ['ce_net_stance', 'bi_mean_sim']

for col, (split, title) in enumerate(plot_items):
    data   = stream_c[split]
    labels = data['labels']
    fnames = data['feat_names']

    for row, feat in enumerate(features_to_plot):
        ax  = axes[row, col]
        idx = fnames.index(feat)
        vals = data['features'][:, idx]
        ax.hist(vals[labels==0], bins=40, alpha=0.6,
                color='#2196f3', label='Genuine', density=True)
        ax.hist(vals[labels==1], bins=40, alpha=0.6,
                color='#e65100', label='OOC/Fake', density=True)
        try:
            auc = roc_auc_score(labels, vals)
            auc = max(auc, 1-auc)
        except Exception:
            auc = 0.5
        ax.set_title(f'{title}\n{feat}  (AUC={auc:.3f})', fontsize=9)
        ax.legend(fontsize=7)
        ax.set_xlabel(feat, fontsize=8)

plt.suptitle('Stream C — Stance Feature Distributions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig_path = FIG_DIR / 'stream_c_stance_distributions.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {fig_path}")

## 12. Summary

In [ ]:
print("=" * 60)
print("STREAM C — stance_scoring.ipynb COMPLETE")
print("=" * 60)
print(f"  Models: cross-encoder/nli-deberta-v3-small + all-MiniLM-L6-v2")
print(f"  Top-K for cross-encoding: {TOP_K_STANCE}")
print(f"  Feature dim: {len(ALL_FEAT_NAMES)} "
      f"(10 evidence + 8 cross-encoder NLI + 3 bi-encoder sim)")
print()
print("  Stream C LR probe AUROC:")
for split in ('val', 'test', 'verite', 'mmfake'):
    X = stream_c[split]['features']
    y = stream_c[split]['labels']
    probs = pipe.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, probs)
    print(f"    {split:8s}: {auc:.4f}")
print()
print("  Saved files:")
for f in sorted(STANCE_DIR.iterdir()):
    print(f"    {f.name:40s} {f.stat().st_size/1e6:6.1f} MB")
print()
print("  Next notebook: stream_c_results.ipynb")